# 🎙️ Whose Dime — voice

**One video, a few runs.** The notebook reads `02_script_voice.txt` in the video's Drive folder, has the narrator say it (Qwen3-TTS) **one block of 2-4 sentences at a time**, and writes in `voice/`:

| File | What it holds |
|---|---|
| `listen_20s.wav`, `listen_20s.json` | the 20 s listening sample (`02c_listen_20s.txt`), made **before** the full voice (decision D6) |
| `voice.wav` | the whole voice track, pauses by punctuation (250 / 450 / 700 ms) |
| `words.json` | the time of every word, block and sentence (the assembly aligns `03_scenes.md` on it) |
| `state.json` | the last full run: `running`, `ready` or `failed`, both speaking rates, the dry endings flagged |
| `blind_test/` | only with `MODE = "blind_test"`: the pairs, `vote.md`, `key.json` (decision D7) |

All the numbers (pauses, detector, rates, seeds) are in `whosedime/notebooks/voice_settings.json` in the repo, not here.

## How to use

1. **Runtime → Change runtime type → T4 GPU.** Fill in the form, then **Runtime → Run all**, and allow Google Drive.
2. **First time only: the narrator** (decision D8). `MODE = "design"`: three candidate voices A, B, C land in `WhoseDime/Voice/candidates/`. Listen, then run again with `MODE = "design"` and `KEEP_VOICE = "A"` (your choice). It becomes `Voice/ref.wav`, cloned by the Base model for every video.
3. **Each video: the 20 s sample.** `MODE = "sample"`, then listen to `voice/listen_20s.wav`.
4. **The full voice.** If the sample sounds right: `MODE = "full"` and tick `I_LISTENED`. About 20-30 min on a T4 (assumed). If Colab stops, run again: finished blocks are kept in `voice/cache/`.
5. A block that sounds wrong: put its number in `REDO_BLOCKS` (`12, 40`; the numbers are in `words.json` and the run's log) and run `full` again.
6. **Video 1 only: the blind test** (D7). `MODE = "blind_test"`: vote in `voice/blind_test/vote.md`, then open `key.json`.

The notebook refuses to start if the checker blocks the voice version (registry unsigned or changed, a drift from the script: decision D4).

In [ ]:
# ── Cell 1: settings ─────────────────────────────────────────────────────────
VIDEO = ""  # @param {type:"string"}
# Empty = the newest video with a 02_script_voice.txt and no ready voice. Else the folder name: "01_uber".

MODE = "sample"  # @param ["design", "sample", "full", "blind_test"]
# design: the narrator (first time) · sample: the 20 s sample · full: the whole voice · blind_test: video 1 only

I_LISTENED = False  # @param {type:"boolean"}
# Tick after listening to voice/listen_20s.wav. "full" refuses to start without it (D6).

KEEP_VOICE = ""  # @param ["", "A", "B", "C"]
# With MODE = "design": the candidate to keep as the narrator. Empty = make the candidates.

REDO_BLOCKS = ""  # @param {type:"string"}
# Block numbers to say again with another seed: "12, 40". Empty = none.

ROOT = "/content/drive/MyDrive/WhoseDime"  # @param {type:"string"}
# The WhoseDime folder in your Drive. If it is "shared with me", add a shortcut to it in "My Drive".

In [ ]:
# ── Cell 2: code, packages, Drive ───────────────────────────────────────────
import os, subprocess, sys
if not globals().get("REPO_READY"):  # opened directly, not through the launcher: take main (D5)
    subprocess.run(["rm", "-rf", "/content/Channel"], check=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                       "https://github.com/Johnfrancis6/Channel.git", "/content/Channel"]).returncode:
        raise SystemExit("❌ Could not fetch the code (branch main).")
sys.path.insert(0, "/content/Channel/whosedime/notebooks")
for name in ("voice_whosedime", "launcher"):
    sys.modules.pop(name, None)
import launcher
import voice_whosedime as vw
COMMIT = globals().get("COMMIT") or launcher.commit()
print(f"✓ Code: main @ {COMMIT}")

for package in ["qwen-tts", "faster-whisper", "librosa", "soundfile", "pyloudnorm"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], capture_output=True, text=True)
    print(("✓ " if r.returncode == 0 else "✗ ") + package + ("" if r.returncode == 0 else "\n" + r.stderr[-500:]))

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

from pathlib import Path
import torch
S = vw.load_settings()
ROOT = Path(ROOT)
if not ROOT.is_dir():
    raise SystemExit(f"❌ No WhoseDime folder at {ROOT}. Fix ROOT in cell 1.")
VOICE_DIR = ROOT / S["drive"]["voice"]
VOICE_DIR.mkdir(exist_ok=True)
if not torch.cuda.is_available():
    print("⚠️  No GPU: Runtime → Change runtime type → T4 GPU (otherwise 5 to 10 times slower).")
print(f"✓ WhoseDime: {ROOT}  ·  mode: {MODE}")

In [ ]:
# ── Cell 3: the narrator — VoiceDesign, saved as the reference (D8) ─────────
# The three descriptions come from whosedime/research/voice_direction.md (A = the one to try first).
if MODE == "design":
    try:
        if KEEP_VOICE:
            VOICE_ID = vw.keep_candidate(VOICE_DIR, KEEP_VOICE, S)
            print("✅ Narrator kept. Next: MODE = 'sample' for the video.")
        else:
            designer = vw.qwen_designer(S)
            vw.design_candidates(designer, VOICE_DIR, S)
            print(f"✅ Candidates in {VOICE_DIR / 'candidates'}: listen to A, B, C, then run again "
                  "with KEEP_VOICE set to your choice.")
    except vw.GateError as e:
        raise SystemExit(f"❌ {e}")
else:
    print("· Narrator: already chosen (MODE is not 'design')")

In [ ]:
# ── Cell 4: the clone on the Base model, and Whisper for the timings ───────
if MODE in ("sample", "full", "blind_test"):
    try:
        VOICE_ID = vw.voice_id(VOICE_DIR)
        FOLDER = vw.find_video(ROOT, VIDEO, S)
    except vw.GateError as e:
        raise SystemExit(f"❌ {e}")
    print(f"🎬 Video: {FOLDER.name}  ·  narrator {VOICE_ID}")
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()  # hides "Setting pad_token_id...", repeated on every block
    synth = vw.qwen_clone_synth(VOICE_DIR, S)
    aligner = vw.whisper_aligner(S)
    print("✓ Models loaded")

In [ ]:
# ── Cell 5: the 20 s listening sample (D6) ──────────────────────────────────
if MODE == "sample":
    try:
        record = vw.run_sample(FOLDER, synth, aligner, S, vid=VOICE_ID, commit=COMMIT)
    except vw.GateError as e:
        raise SystemExit(f"❌ {e}")
    from IPython.display import Audio, display
    display(Audio(str(FOLDER / "voice" / S["files"]["sample_wav"])))
    print("✅ Listen. If it sounds right: MODE = 'full', tick I_LISTENED, run all.")
    print("   If not, tell Claude what is wrong (pauses, pace, a word): the settings change in the repo.")

In [ ]:
# ── Cell 6: the whole voice ─────────────────────────────────────────────────
if MODE == "full":
    try:
        state = vw.run_full(FOLDER, synth, aligner, S, vid=VOICE_ID, listened=I_LISTENED,
                            redo=REDO_BLOCKS, commit=COMMIT)
    except vw.GateError as e:
        raise SystemExit(f"❌ {e}")
    print(f"\n✅ Voice ready in Videos/{FOLDER.name}/voice/ — the skill reads state.json "
          "(rate_total_wpm goes to Memory/lessons.md).")

In [ ]:
# ── Cell 7: option, the blind test of D7 (video 1) ──────────────────────────
if MODE == "blind_test":
    key = vw.run_blind_test(FOLDER, synth, aligner, S, vid=VOICE_ID, commit=COMMIT)
    print("✅ Vote in voice/blind_test/vote.md (X or Y for each pair), then open key.json.")